## DSE 230- Spark Streaming Demo

* Spark Structured Streaming is a scalable and fault-tolerant stream processing engine built on top of the Spark SQL engine. It enables the processing of real-time streaming data using the same API as the batch processing engine of Spark.

* Structured Streaming provides high-level APIs for processing streaming data in a batch-like fashion. It abstracts away the complexities of distributed streaming processing and enables developers to write continuous queries on the data streams, just like writing batch queries on static data.

* Under the hood, Structured Streaming treats streaming data as a continuous table, and each arriving data record is treated as a new row in the table. Developers can use the same APIs and SQL syntax to manipulate and process the data as they would for batch processing.

* Structured Streaming provides a range of transformations and actions that can be used to process the data, such as filter, map, groupBy, window, join, and more. It also supports various input sources, such as Kafka, file-based systems, and socket-based streams.

* Structured Streaming is designed to handle both high-volume and low-latency data streams and can be easily scaled up or down based on the processing requirements. It also provides end-to-end fault-tolerance, which ensures that the data processing is resilient to failures.

#### Quick example
(Taken from Spark's Official Website-https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html )

Count the number of words received from a data server listening on a TCP socket

1. Open a new terminal window in Jupyter Lab and run:
    * apt install netcat
    * nc -lk 9999
2. Run the notebook cells
3. Enter an input statement in the window and watch it processed here

To begin utilizing Spark's functionalities, we must initially import the required classes and establish a local SparkSession, which serves as the foundation for all Spark-related tasks.

In [ ]:
# Suppress native-hadoop warning
!sed -i '$a\# Add the line for suppressing the NativeCodeLoader warning \nlog4j.logger.org.apache.hadoop.util.NativeCodeLoader=ERROR,console' /$HADOOP_HOME/etc/hadoop/log4j.properties
import os; os.close(os.dup2(os.open(os.devnull, os.O_WRONLY), 2))

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import explode
from pyspark.sql.functions import split, desc

spark = SparkSession \
    .builder \
    .appName("StructuredNetworkWordCount") \
    .getOrCreate()

Next, let’s create a streaming DataFrame that represents text data received from a server listening on localhost:9999, and transform the DataFrame to calculate word counts.

In [ ]:
# Create DataFrame representing the stream of input lines from connection to localhost:9999
lines = spark \
    .readStream \
    .format("socket") \
    .option("host", "localhost") \
    .option("port", 9999) \
    .load()

# Split the lines into words
words = lines.select(
   explode(
       split(lines.value, " ")
   ).alias("word")
)

# Generate running word count
wordCounts = words.groupBy("word").count().orderBy(desc('count'))

This lines DataFrame represents an unbounded table containing the streaming text data. This table contains one column of strings named “value”, and each line in the streaming text data becomes a row in the table. Note, that this is not currently receiving any data as we are just setting up the transformation, and have not yet started it. Next, we have used two built-in SQL functions - split and explode, to split each line into multiple rows with a word each. In addition, we use the function alias to name the new column as “word”. Finally, we have defined the wordCounts DataFrame by grouping by the unique values in the Dataset and counting them. Note that this is a streaming DataFrame which represents the running word counts of the stream.

We have now set up the query on the streaming data. All that is left is to actually start receiving data and computing the counts. To do this, we set it up to print the complete set of counts (specified by outputMode("complete")) to the console every time they are updated. And then start the streaming computation using start().

You can read more about the different output modes [here](https://www.databricks.com/spark/getting-started-with-apache-spark/streaming)

In [ ]:
# Start running the query that prints the running counts to the console
query = wordCounts \
    .writeStream \
    .outputMode("complete") \
    .format("console") \
    .start()

After this code is executed, the streaming computation will have started in the background. The query object is a handle to that active streaming query, and we have decided to wait for the termination of the query using awaitTermination() to prevent the process from exiting while the query is active.

In [ ]:
query.stop()